In [24]:
# 1 — Imports

import os
import time
import calendar
import requests
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

In [25]:
# 2 — Configuración y conexión

# ── Cargar variables de entorno ──────────────────────────────
load_dotenv()

DB_SERVER   = os.getenv("DB_SERVER")
DB_NAME     = os.getenv("DB_NAME")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

# ── Verificar que todas las variables están presentes ────────
variables = {
    "DB_SERVER"  : DB_SERVER,
    "DB_NAME"    : DB_NAME,
    "DB_USER"    : DB_USER,
    "DB_PASSWORD": DB_PASSWORD
}

faltantes = [k for k, v in variables.items() if not v]

if faltantes:
    raise ValueError(f"Variables de entorno faltantes: {faltantes}")

print("Variables de entorno cargadas correctamente")

# ── Conexión ─────────────────────────────────────────────────
start  = time.time()
engine = create_engine(
    f"mssql+pyodbc://{DB_USER}:{DB_PASSWORD}@{DB_SERVER}/{DB_NAME}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

with engine.connect() as conn:
    elapsed = round(time.time() - start, 2)
    print(f"Conexión exitosa | Tiempo: {elapsed}s")

Variables de entorno cargadas correctamente
Conexión exitosa | Tiempo: 0.56s


In [26]:
# 3 — Funciones auxiliares

def verificar_api(url, params):
    """Verifica que el endpoint responde correctamente antes de procesar."""
    start    = time.time()
    response = requests.get(url, headers = {"Accept": "application/json"}, params = params)

    if response.status_code != 200:
        raise ConnectionError(f"API respondió con status {response.status_code}")
    if not response.text.strip():
        raise ValueError("API respondió con cuerpo vacío")

    return response.json()


def cargar_tabla(df, tabla, anio, mes = None):
    """Elimina el rango e inserta el DataFrame en la tabla indicada."""
    start = time.time()

    if mes:
        ultimo_dia = calendar.monthrange(anio, mes)[1]
        rango_ini  = f"{anio}-{mes:02d}-01"
        rango_fin  = f"{anio}-{mes:02d}-{ultimo_dia}"
    else:
        rango_ini  = f"{anio}-01-01"
        rango_fin  = f"{anio}-12-31"

    with engine.connect() as conn:
        conn.execute(text(f"""
            DELETE FROM {tabla}
            WHERE Fecha BETWEEN :ini AND :fin
        """), {"ini": rango_ini, "fin": rango_fin})
        conn.commit()

    df.to_sql(tabla, con = engine, if_exists = "append", index = False)

    elapsed = round(time.time() - start, 2)
    return elapsed

In [27]:
# 4 — Generacion

print("=" * 60)
print("GENERACION")
print("=" * 60)

start_total = time.time()
total_filas = 0
anios       = range(2014, datetime.now().year + 1)
url         = "https://apidatos.ree.es/es/datos/generacion/estructura-generacion"

verificar_api(url, {
    "start_date": "2024-01-01T00:00",
    "end_date"  : "2024-01-02T23:59",
    "time_trunc": "day",
    "geo_limit" : "peninsular",
    "geo_ids"   : "8741"
})

for anio in anios:
    start_iter = time.time()
    params     = {
        "start_date": f"{anio}-01-01T00:00",
        "end_date"  : f"{anio}-12-31T23:59",
        "time_trunc": "day",
        "geo_limit" : "peninsular",
        "geo_ids"   : "8741"
    }

    try:
        datos = verificar_api(url, params).get("included", [])
    except Exception as e:
        print(f"{anio}: error — {e}")
        continue

    registros = []
    for fuente in datos:
        tipo = fuente["attributes"]["title"]
        for punto in fuente["attributes"]["values"]:
            registros.append({
                "Fecha"     : punto["datetime"][:10],
                "Fuente"    : tipo,
                "Valor_mwh" : punto["value"],
                "Porcentaje": punto["percentage"]
            })

    if not registros:
        print(f"{anio}: sin datos")
        continue

    df          = pd.DataFrame(registros)
    df["Fecha"] = pd.to_datetime(df["Fecha"]).dt.date

    elapsed      = cargar_tabla(df, "Generacion", anio)
    total_filas += len(df)
    print(f"{anio}: {len(df):,} filas | {elapsed}s")

print(f"\nTotal Generacion: {total_filas:,} filas | Tiempo total: {round(time.time() - start_total, 2)}s")

GENERACION
2014: 4,669 filas | 1.23s
2015: 4,386 filas | 1.14s
2016: 4,398 filas | 1.15s
2017: 4,386 filas | 1.19s
2018: 4,386 filas | 1.14s
2019: 4,385 filas | 1.14s
2020: 4,400 filas | 1.21s
2021: 4,385 filas | 1.14s
2022: 4,384 filas | 1.14s
2023: 4,381 filas | 1.12s
2024: 4,396 filas | 1.12s
2025: 4,548 filas | 1.14s
2026: 1,026 filas | 0.52s

Total Generacion: 54,130 filas | Tiempo total: 90.74s


In [28]:
# 5 — Demanda

print("=" * 60)
print("DEMANDA")
print("=" * 60)

start_total = time.time()
total_filas = 0
url         = "https://apidatos.ree.es/es/datos/demanda/evolucion"

verificar_api(url, {
    "start_date": "2024-01-01T00:00",
    "end_date"  : "2024-01-02T23:59",
    "time_trunc": "day",
    "geo_limit" : "peninsular",
    "geo_ids"   : "8741"
})

for anio in anios:
    start_iter = time.time()
    params     = {
        "start_date": f"{anio}-01-01T00:00",
        "end_date"  : f"{anio}-12-31T23:59",
        "time_trunc": "day",
        "geo_limit" : "peninsular",
        "geo_ids"   : "8741"
    }

    try:
        datos = verificar_api(url, params).get("included", [])
    except Exception as e:
        print(f"{anio}: error — {e}")
        continue

    registros = []
    for item in datos:
        tipo = item["attributes"]["title"]
        for punto in item["attributes"]["values"]:
            registros.append({
                "Fecha"    : punto["datetime"][:10],
                "Tipo"     : tipo,
                "Valor_mwh": punto["value"]
            })

    if not registros:
        print(f"{anio}: sin datos")
        continue

    df          = pd.DataFrame(registros)
    df["Fecha"] = pd.to_datetime(df["Fecha"]).dt.date

    elapsed      = cargar_tabla(df, "Demanda", anio)
    total_filas += len(df)
    print(f"{anio}: {len(df):,} filas | {elapsed}s")

print(f"\nTotal Demanda: {total_filas:,} filas | Tiempo total: {round(time.time() - start_total, 2)}s")

DEMANDA
2014: 365 filas | 0.42s
2015: 365 filas | 0.42s
2016: 366 filas | 0.47s
2017: 365 filas | 0.42s
2018: 365 filas | 0.41s
2019: 365 filas | 0.42s
2020: 366 filas | 0.42s
2021: 365 filas | 0.41s
2022: 365 filas | 0.41s
2023: 365 filas | 0.42s
2024: 366 filas | 0.43s
2025: 365 filas | 0.41s
2026: 79 filas | 0.43s

Total Demanda: 4,462 filas | Tiempo total: 86.93s


In [29]:
# 6 — Emisiones

print("=" * 60)
print("EMISIONES")
print("=" * 60)

start_total = time.time()
total_filas = 0
url         = "https://apidatos.ree.es/es/datos/generacion/estructura-generacion-emisiones-asociadas"

verificar_api(url, {
    "start_date": "2024-01-01T00:00",
    "end_date"  : "2024-01-02T23:59",
    "time_trunc": "day",
    "geo_limit" : "peninsular",
    "geo_ids"   : "8741"
})

for anio in anios:
    start_iter = time.time()
    params     = {
        "start_date": f"{anio}-01-01T00:00",
        "end_date"  : f"{anio}-12-31T23:59",
        "time_trunc": "day",
        "geo_limit" : "peninsular",
        "geo_ids"   : "8741"
    }

    try:
        datos = verificar_api(url, params).get("included", [])
    except Exception as e:
        print(f"{anio}: error — {e}")
        continue

    registros = []
    for item in datos:
        tipo = item["attributes"]["title"]
        for punto in item["attributes"]["values"]:
            registros.append({
                "Fecha": punto["datetime"][:10],
                "Tipo" : tipo,
                "Valor": punto["value"]
            })

    if not registros:
        print(f"{anio}: sin datos")
        continue

    df          = pd.DataFrame(registros)
    df["Fecha"] = pd.to_datetime(df["Fecha"]).dt.date

    elapsed      = cargar_tabla(df, "Emisiones", anio)
    total_filas += len(df)
    print(f"{anio}: {len(df):,} filas | {elapsed}s")

print(f"\nTotal Emisiones: {total_filas:,} filas | Tiempo total: {round(time.time() - start_total, 2)}s")

EMISIONES
2014: 4,304 filas | 0.94s
2015: 4,021 filas | 0.91s
2016: 4,032 filas | 0.93s
2017: 4,021 filas | 0.88s
2018: 4,021 filas | 0.88s
2019: 4,020 filas | 0.93s
2020: 4,034 filas | 0.9s
2021: 4,020 filas | 0.89s
2022: 4,019 filas | 0.9s
2023: 4,016 filas | 0.94s
2024: 4,030 filas | 0.92s
2025: 4,183 filas | 0.89s
2026: 947 filas | 0.55s

Total Emisiones: 49,668 filas | Tiempo total: 19.88s


In [30]:
# 7 — Precios

print("=" * 60)
print("PRECIOS")
print("=" * 60)

start_total = time.time()
total_filas = 0
url         = "https://apidatos.ree.es/es/datos/mercados/precios-mercados-tiempo-real"

verificar_api(url, {
    "start_date": "2024-01-01T00:00",
    "end_date"  : "2024-01-02T23:59",
    "time_trunc": "hour"
})

for anio in range(2014, datetime.now().year + 1):
    for mes in range(1, 13):

        if datetime(anio, mes, 1) > datetime.now():
            continue

        ultimo_dia = calendar.monthrange(anio, mes)[1]
        params     = {
            "start_date": f"{anio}-{mes:02d}-01T00:00",
            "end_date"  : f"{anio}-{mes:02d}-{ultimo_dia}T23:59",
            "time_trunc": "hour"
        }

        try:
            datos = verificar_api(url, params).get("included", [])
        except Exception as e:
            print(f"{anio}-{mes:02d}: error — {e}")
            continue

        registros = []
        for item in datos:
            tipo = item["attributes"]["title"]
            for punto in item["attributes"]["values"]:
                dt   = punto["datetime"]
                zona = dt[23:29]
                registros.append({
                    "Fecha"        : dt[:10],
                    "Hora"         : dt[11:19],
                    "Zona"         : zona,
                    "Tipo"         : tipo,
                    "Valor_eur_mwh": punto["value"]
                })

        if not registros:
            print(f"{anio}-{mes:02d}: sin datos")
            continue

        df          = pd.DataFrame(registros)
        df["Fecha"] = pd.to_datetime(df["Fecha"]).dt.date
        df["Hora"]  = pd.to_datetime(df["Hora"], format = "%H:%M:%S").dt.time

        elapsed      = cargar_tabla(df, "Precios", anio, mes)
        total_filas += len(df)

    print(f"{anio}: acumulado {total_filas:,} filas")

print(f"\nTotal Precios: {total_filas:,} filas | Tiempo total: {round(time.time() - start_total, 2)}s")

PRECIOS
2014: acumulado 8,760 filas
2015: acumulado 17,520 filas
2016: acumulado 26,304 filas
2017: acumulado 35,064 filas
2018: acumulado 43,824 filas
2019: acumulado 52,584 filas
2020: acumulado 61,368 filas
2021: acumulado 75,265 filas
2022: acumulado 92,785 filas
2023: acumulado 110,305 filas
2024: acumulado 127,873 filas
2025: acumulado 171,673 filas
2026: acumulado 181,273 filas

Total Precios: 181,273 filas | Tiempo total: 313.17s


In [31]:
# 8 — Intercambios

print("=" * 60)
print("INTERCAMBIOS")
print("=" * 60)

start_total = time.time()
total_filas = 0
url         = "https://apidatos.ree.es/es/datos/intercambios/todas-fronteras-programados"

verificar_api(url, {
    "start_date": "2024-01-01T00:00",
    "end_date"  : "2024-01-02T23:59",
    "time_trunc": "day",
    "geo_limit" : "peninsular",
    "geo_ids"   : "8741"
})

for anio in anios:
    start_iter = time.time()
    params     = {
        "start_date": f"{anio}-01-01T00:00",
        "end_date"  : f"{anio}-12-31T23:59",
        "time_trunc": "day",
        "geo_limit" : "peninsular",
        "geo_ids"   : "8741"
    }

    try:
        datos = verificar_api(url, params).get("included", [])
    except Exception as e:
        print(f"{anio}: error — {e}")
        continue

    registros = []
    for pais_item in datos:
        pais    = pais_item["attributes"]["title"].capitalize()
        content = pais_item["attributes"].get("content", [])
        for flujo in content:
            tipo = flujo["attributes"]["title"]
            for punto in flujo["attributes"]["values"]:
                registros.append({
                    "Fecha"    : punto["datetime"][:10],
                    "Pais"     : pais,
                    "Tipo"     : tipo,
                    "Valor_mwh": punto["value"]
                })

    if not registros:
        print(f"{anio}: sin datos")
        continue

    df          = pd.DataFrame(registros)
    df["Fecha"] = pd.to_datetime(df["Fecha"]).dt.date

    elapsed      = cargar_tabla(df, "Intercambios", anio)
    total_filas += len(df)
    print(f"{anio}: {len(df):,} filas | {elapsed}s")

print(f"\nTotal Intercambios: {total_filas:,} filas | Tiempo total: {round(time.time() - start_total, 2)}s")

INTERCAMBIOS
2014: 3,435 filas | 1.1s
2015: 3,338 filas | 0.97s
2016: 3,562 filas | 1.01s
2017: 3,486 filas | 1.01s
2018: 3,407 filas | 1.01s
2019: 3,570 filas | 1.04s
2020: 3,391 filas | 1.01s
2021: 3,472 filas | 1.01s
2022: 3,740 filas | 1.12s
2023: 3,805 filas | 1.08s
2024: 3,761 filas | 1.05s
2025: 3,751 filas | 1.06s
2026: 819 filas | 0.54s

Total Intercambios: 43,537 filas | Tiempo total: 49.61s


In [32]:
# 9 — Resumen final

print("=" * 60)
print("RESUMEN FINAL")
print("=" * 60)

start = time.time()

with engine.connect() as conn:
    resultado  = conn.execute(text("""
        SELECT 'Generacion'    AS Tabla, COUNT(*) AS Filas, MIN(Fecha) AS Desde, MAX(Fecha) AS Hasta FROM Generacion
        UNION ALL SELECT 'Demanda',      COUNT(*), MIN(Fecha), MAX(Fecha) FROM Demanda
        UNION ALL SELECT 'Emisiones',    COUNT(*), MIN(Fecha), MAX(Fecha) FROM Emisiones
        UNION ALL SELECT 'Precios',      COUNT(*), MIN(Fecha), MAX(Fecha) FROM Precios
        UNION ALL SELECT 'Intercambios', COUNT(*), MIN(Fecha), MAX(Fecha) FROM Intercambios
    """))
    df_resumen = pd.DataFrame(resultado.fetchall(), columns = ["Tabla", "Filas", "Desde", "Hasta"])

print(df_resumen.to_string(index = False))
print(f"\nTiempo consulta: {round(time.time() - start, 2)}s")

RESUMEN FINAL
       Tabla  Filas      Desde      Hasta
  Generacion  54130 2014-01-01 2026-03-20
     Demanda   4462 2014-01-01 2026-03-20
   Emisiones  49668 2014-01-01 2026-03-20
     Precios 181273 2014-01-01 2026-03-21
Intercambios  43537 2014-01-01 2026-03-20

Tiempo consulta: 0.14s
